# Red Neuronal LSTM para Mantenimiento Predictivo
## Asignatura: Aprendizaje Automático

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI ML Repository)

**Objetivo:** Construir, entrenar y evaluar un modelo LSTM para predecir fallos en maquinaria industrial, transformando los datos en ventanas temporales.

**Aplicación:**
- Predicción de fallo futuro
- Mantenimiento predictivo

---

## 1. Instalación de Dependencias

In [ ]:
!pip install ucimlrepo tensorflow scikit-learn pandas numpy matplotlib seaborn

## 2. Importación de Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 3. Carga del Dataset

In [ ]:
from ucimlrepo import fetch_ucirepo

# Cargar dataset desde UCI ML Repository
ai4i_2020_predictive_maintenance_dataset = fetch_ucirepo(id=601)

X = ai4i_2020_predictive_maintenance_dataset.data.features
y = ai4i_2020_predictive_maintenance_dataset.data.targets

print("Dataset cargado exitosamente.")
print(f"Dimensiones de X (features): {X.shape}")
print(f"Dimensiones de y (targets): {y.shape}")

## 4. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Vista general del dataset
print("=" * 60)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)
print(f"\nNúmero de muestras: {X.shape[0]}")
print(f"Número de features: {X.shape[1]}")
print(f"\nColumnas de features: {list(X.columns)}")
print(f"Columnas de targets: {list(y.columns)}")

In [ ]:
# Primeras filas
print("\nPrimeras 5 filas de X:")
X.head()

In [ ]:
print("\nPrimeras 5 filas de y:")
y.head()

In [ ]:
# Tipos de datos y valores nulos
print("\nTipos de datos en X:")
print(X.dtypes)
print(f"\nValores nulos en X:\n{X.isnull().sum()}")
print(f"\nValores nulos en y:\n{y.isnull().sum()}")

In [ ]:
# Estadísticas descriptivas
X.describe()

In [ ]:
# Distribución de la variable objetivo (Machine failure)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
target_col = 'Machine failure'
counts = y[target_col].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(counts.index.astype(str), counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribución de Machine Failure', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Clase (0=No Fallo, 1=Fallo)')
axes[0].set_ylabel('Frecuencia')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Gráfico circular
axes[1].pie(counts.values, labels=['No Fallo', 'Fallo'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.1),
            shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Proporción de Fallos', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nClase 0 (No Fallo): {counts[0]} ({counts[0]/len(y)*100:.1f}%)")
print(f"Clase 1 (Fallo): {counts[1]} ({counts[1]/len(y)*100:.1f}%)")
print(f"Ratio de desbalance: {counts[0]/counts[1]:.1f}:1")

In [ ]:
# Distribución de las features numéricas
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        axes[i].hist(X[col], bins=40, color='#3498db', edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Distribución de {col}', fontsize=12, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frecuencia')
        axes[i].axvline(X[col].mean(), color='red', linestyle='--', label=f'Media={X[col].mean():.1f}')
        axes[i].legend()

# Ocultar ejes sobrantes
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Features Numéricas', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(12, 8))

# Combinar features y target para la correlación
df_corr = pd.concat([X[numeric_cols], y[target_col]], axis=1)
corr_matrix = df_corr.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por clase de fallo
df_full = pd.concat([X, y[target_col]], axis=1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        df_full.boxplot(column=col, by=target_col, ax=axes[i])
        axes[i].set_title(f'{col} por Clase', fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Machine Failure')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Features Numéricas por Clase de Fallo', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Preprocesamiento de Datos

In [ ]:
# Variable objetivo: Machine failure (clasificación binaria)
y_target = y[target_col].values

print(f"Variable objetivo seleccionada: '{target_col}'")
print(f"Valores únicos: {np.unique(y_target)}")
print(f"Distribución: 0={np.sum(y_target==0)}, 1={np.sum(y_target==1)}")

In [ ]:
# Codificación de variables categóricas
X_processed = X.copy()

# Identificar columnas categóricas
cat_cols = X_processed.select_dtypes(include=['object']).columns.tolist()
print(f"Columnas categóricas encontradas: {cat_cols}")

# Codificar con LabelEncoder
le = LabelEncoder()
for col in cat_cols:
    print(f"  - {col}: {X_processed[col].unique()}")
    X_processed[col] = le.fit_transform(X_processed[col])

print(f"\nDimensiones después de codificación: {X_processed.shape}")
X_processed.head()

In [ ]:
# Normalización de features con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

print(f"Shape después de escalar: {X_scaled.shape}")
print(f"Media (aprox 0): {X_scaled.mean(axis=0).round(4)}")
print(f"Std  (aprox 1): {X_scaled.std(axis=0).round(4)}")

## 6. Creación de Ventanas Temporales para LSTM

Las redes LSTM esperan datos en formato secuencial. Transformamos los datos en ventanas temporales deslizantes:

```
X(t-5), X(t-4), X(t-3), X(t-2), X(t-1) → fallo en t
```

Cada ventana contiene `n_steps` pasos temporales consecutivos para predecir el fallo en el siguiente instante.

In [ ]:
def create_sequences(X, y, n_steps=5):
    """
    Transforma los datos en ventanas temporales deslizantes para LSTM.
    
    Parámetros:
        X: array de features (n_samples, n_features)
        y: array de etiquetas (n_samples,)
        n_steps: número de pasos temporales por ventana
    
    Retorna:
        X_seq: array (n_samples - n_steps, n_steps, n_features)
        y_seq: array (n_samples - n_steps,)
    """
    X_seq, y_seq = [], []
    for i in range(n_steps, len(X)):
        X_seq.append(X[i - n_steps:i])  # Ventana de n_steps
        y_seq.append(y[i])               # Etiqueta en el instante t
    return np.array(X_seq), np.array(y_seq)

# Definir tamaño de ventana temporal
N_STEPS = 5

X_seq, y_seq = create_sequences(X_scaled, y_target, n_steps=N_STEPS)

print(f"Tamaño de ventana temporal (n_steps): {N_STEPS}")
print(f"Shape de X secuencial: {X_seq.shape}  →  (muestras, pasos_temporales, features)")
print(f"Shape de y secuencial: {y_seq.shape}")
print(f"\nEjemplo de una ventana (primeros 2 features):")
print(X_seq[0, :, :2])

In [ ]:
# División en conjuntos de entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras")
print(f"\nDistribución en entrenamiento: No Fallo={np.sum(y_train==0)}, Fallo={np.sum(y_train==1)}")
print(f"Distribución en prueba:        No Fallo={np.sum(y_test==0)}, Fallo={np.sum(y_test==1)}")

In [ ]:
# Calcular pesos de clase para manejar el desbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print(f"Pesos de clase calculados (para manejar desbalance):")
print(f"  Clase 0 (No Fallo): {class_weight_dict[0]:.4f}")
print(f"  Clase 1 (Fallo):    {class_weight_dict[1]:.4f}")

## 7. Construcción del Modelo LSTM

In [ ]:
def build_lstm_model(n_steps, n_features):
    """
    Construye una red LSTM para clasificación binaria.
    
    Arquitectura:
        - LSTM (64 unidades) + BatchNorm + Dropout
        - LSTM (32 unidades) + BatchNorm + Dropout
        - Dense (16 unidades, ReLU)
        - Dense (1 unidad, Sigmoid) → salida binaria
    """
    model = Sequential([
        # Primera capa LSTM
        LSTM(64, return_sequences=True, input_shape=(n_steps, n_features)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Segunda capa LSTM
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),
        
        # Capas densas
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Construir el modelo
n_features = X_train.shape[2]
model = build_lstm_model(N_STEPS, n_features)

# Resumen de la arquitectura
model.summary()

In [ ]:
# Visualizar la arquitectura del modelo
print("\n" + "=" * 60)
print("ARQUITECTURA DEL MODELO LSTM")
print("=" * 60)
print(f"\nInput shape:  ({N_STEPS}, {n_features})  →  {N_STEPS} pasos temporales × {n_features} features")
print(f"Output shape: (1,)  →  Probabilidad de fallo (sigmoid)")
print(f"\nTotal parámetros: {model.count_params():,}")
print(f"Función de pérdida: Binary Crossentropy")
print(f"Optimizador: Adam (lr=0.001)")

## 8. Entrenamiento del Modelo

In [ ]:
# Callbacks para el entrenamiento
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Entrenar el modelo
print("Iniciando entrenamiento...\n")

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\nEntrenamiento completado.")

## 9. Visualización del Entrenamiento

In [ ]:
# Curvas de pérdida y accuracy
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pérdida (Loss)
axes[0].plot(history.history['loss'], label='Entrenamiento', linewidth=2, color='#3498db')
axes[0].plot(history.history['val_loss'], label='Validación', linewidth=2, color='#e74c3c')
axes[0].set_title('Curva de Pérdida (Loss)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Entrenamiento', linewidth=2, color='#3498db')
axes[1].plot(history.history['val_accuracy'], label='Validación', linewidth=2, color='#e74c3c')
axes[1].set_title('Curva de Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Historial de Entrenamiento del Modelo LSTM', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Evaluación del Modelo

In [ ]:
# Evaluar en el conjunto de prueba
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("=" * 60)
print("RESULTADOS EN EL CONJUNTO DE PRUEBA")
print("=" * 60)
print(f"\nPérdida (Loss):  {test_loss:.4f}")
print(f"Accuracy:        {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Predicciones
y_pred_proba = model.predict(X_test).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Reporte de clasificación completo
print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
print(classification_report(y_test, y_pred,
                            target_names=['No Fallo (0)', 'Fallo (1)']))

In [ ]:
# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Fallo (0)', 'Fallo (1)'],
            yticklabels=['No Fallo (0)', 'Fallo (1)'],
            linewidths=2, linecolor='white',
            annot_kws={'size': 16, 'fontweight': 'bold'})
plt.title('Matriz de Confusión', fontsize=16, fontweight='bold')
plt.ylabel('Valor Real', fontsize=13)
plt.xlabel('Valor Predicho', fontsize=13)
plt.tight_layout()
plt.show()

# Interpretación
tn, fp, fn, tp = cm.ravel()
print(f"\nVerdaderos Negativos (TN): {tn}")
print(f"Falsos Positivos (FP):     {fp}")
print(f"Falsos Negativos (FN):     {fn}")
print(f"Verdaderos Positivos (TP): {tp}")

In [ ]:
# Curva ROC
fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Curva Precision-Recall
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC
axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
             label=f'LSTM (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='#e74c3c')
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=12)
axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=12)
axes[0].set_title('Curva ROC', fontsize=14, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Precision-Recall
axes[1].plot(recall, precision, color='#2ecc71', linewidth=2.5,
             label=f'LSTM (AUC = {pr_auc:.4f})')
axes[1].fill_between(recall, precision, alpha=0.15, color='#2ecc71')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Curva Precision-Recall', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower left', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Curvas de Evaluación del Modelo', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nÁrea bajo la curva ROC (AUC-ROC): {roc_auc:.4f}")
print(f"Área bajo la curva PR (AUC-PR):   {pr_auc:.4f}")

In [ ]:
# Distribución de probabilidades predichas
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma por clase
axes[0].hist(y_pred_proba[y_test == 0], bins=50, alpha=0.7, color='#2ecc71',
             label='No Fallo (Real)', edgecolor='black')
axes[0].hist(y_pred_proba[y_test == 1], bins=50, alpha=0.7, color='#e74c3c',
             label='Fallo (Real)', edgecolor='black')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=2, label='Umbral (0.5)')
axes[0].set_xlabel('Probabilidad Predicha', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].set_title('Distribución de Probabilidades Predichas', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Boxplot
data_box = [y_pred_proba[y_test == 0], y_pred_proba[y_test == 1]]
bp = axes[1].boxplot(data_box, labels=['No Fallo', 'Fallo'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_ylabel('Probabilidad Predicha', fontsize=12)
axes[1].set_title('Distribución por Clase Real', fontsize=14, fontweight='bold')
axes[1].axhline(0.5, color='black', linestyle='--', linewidth=1, label='Umbral')
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Resumen de Métricas

In [ ]:
# Tabla resumen
from sklearn.metrics import f1_score, precision_score, recall_score

metrics = {
    'Métrica': ['Accuracy', 'Precision (Fallo)', 'Recall (Fallo)',
                'F1-Score (Fallo)', 'AUC-ROC', 'AUC-PR'],
    'Valor': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc,
        pr_auc
    ]
}

df_metrics = pd.DataFrame(metrics)
df_metrics['Valor'] = df_metrics['Valor'].apply(lambda x: f'{x:.4f}')

print("\n" + "=" * 50)
print("   RESUMEN FINAL DE MÉTRICAS")
print("=" * 50)
print(df_metrics.to_string(index=False))
print("=" * 50)

## 12. Conclusiones

### Resumen del proyecto

1. **Dataset:** Se utilizó el AI4I 2020 Predictive Maintenance Dataset con 10,000 muestras y múltiples features de sensores industriales.

2. **Preprocesamiento:** Se codificaron variables categóricas, se normalizaron los datos y se crearon ventanas temporales deslizantes de 5 pasos para alimentar la red LSTM.

3. **Modelo:** Se implementó una red LSTM con dos capas recurrentes (64 y 32 unidades), BatchNormalization y Dropout para regularización.

4. **Desbalance de clases:** Se aplicaron pesos de clase balanceados durante el entrenamiento para contrarrestar la desproporción entre fallos y no fallos.

5. **Ventanas temporales:** La transformación `X(t-5), X(t-4), X(t-3), X(t-2), X(t-1) → fallo en t` permite al modelo capturar patrones secuenciales previos al fallo.

### Aplicaciones prácticas

- **Mantenimiento predictivo:** Anticipar fallos de maquinaria antes de que ocurran.
- **Reducción de costos:** Minimizar paradas no planificadas en entornos industriales.
- **Optimización de recursos:** Planificar mantenimiento basado en datos, no en calendarios fijos.